<a href="https://colab.research.google.com/github/maazali04/deep-learning-from-scratch/blob/main/04_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04: Convolutional Neural Networks (CNNs)

## What is CNN?
A Convolutional Neural Network (CNN) is a special type of deep learning model designed specifically to process grid-like data, most famously digital images.

Before CNNs, standard neural networks struggled with images. They treated pixels as a flat, unorganized list of numbers, completely destroying the spatial context (the relationship between neighboring pixels). A CNN, however, is built to preserve that structure, allowing it to look at an image the same way a human eye does: by detecting edges, shapes, and textures to understand the whole picture.

## Core Building Blocks of CNN
The Core Building Blocks of a CNN:
- Convolutional Layer (nn.Conv2d): Applies small, learnable filter matrices (kernels) across spatial regions. Each filter learns to extract specific features (edges, textures, shapes).
- Pooling Layer (nn.MaxPool2d): Downsamples feature maps (e.g., taking the maximum value in a $2 \times 2$ window) to reduce spatial dimensions, lower compute requirements, and provide translation invariance.
- Fully Connected Layer (nn.Linear): Flattened feature maps pass into final linear layers at the end of the network to output classification logits.

### The Convolution Operation

The Convolution Layer perform Convolution operation.

The core purpose of a convolution layer is to automatically extract spatial features from an input image using a sliding window approach.

- The Components:
  - Input Image: The raw grid of pixels.
  - Kernel / Filter / Detector: A small matrix (commonly $3 \times 3$ or $5 \times 5)$ containing learnable weights that detects specific features like edges, textures, or shapes.
  - Feature Map (Activation Map): The output grid generated after the kernel completes its scan. It highlights exactly where specific features were detected in the image.
  
- The Process:
  - The kernel is placed onto the top-left corner of the image.
  - An element-wise multiplication is performed between each cell of the kernel and the corresponding image pixels beneath it.
  - All resulting products are summed together into a single numerical value.
  - The kernel shifts to the right by one column to calculate the next number.
  - Once a row is completed, the kernel moves down to the next row and repeats the process until the entire image is covered.

### Navigating 3D Spaces: Channels & Multi-Filter Outputs

Images are rarely flat 2D shapes; they have depth. The convolution operation naturally handles this 3D landscape:
- Handling RGB Inputs: A standard color image has 3 input channels (Red, Green, Blue). To match this depth, a single kernel is actually a 3D block sized $(3 \times 3 \times 3)$. It calculates the element-wise sum across all three color channels simultaneously, compressing the result down into a single, 2D Feature Map.
- The Multi-Kernel Stacking Rule: One kernel can only detect one specific pattern. To understand a whole image, a layer uses multiple kernels simultaneously.
- Dimensional Example: If you pass a $(6 \times 6 \times 3)$ image through \(3\) separate kernels (each sized $3 \times 3 \times 3$), you will generate 3 independent 2D feature maps. Stacking them together creates a final output tensor of $4 \times 4 \times 3 (Length \times  Width \times  Number of Kernels)$.
$$\text{Image }(6\times 6\times 3)\;\circledast \;\text{Kernels }(3\times 3\times 3\times \mathbf{3})\;\longrightarrow \;\text{Feature Map }(4\times 4\times \mathbf{3})$$

### Padding (Fixing the Borders)
If you perform standard convolutions repeatedly, you will immediately run into two structural issues:
1. Shrinkage: The feature map naturally becomes smaller than the input image at every layer, restricting how deep your network can go.
2. Edge Information Loss: The corner and edge pixels are only touched by the kernel once, whereas the center pixels are calculated repeatedly. This causes the network to lose critical information along the borders of your images.

**The Fix: Zero Padding**

Padding injects mock rows of zeros to the top and bottom, and mock columns of zeros to the left and right of the image before the kernel begins its scan.

- `padding="valid"` (In Keras/Frameworks): This means no padding is applied. The feature map will naturally shrink.

- `padding="same"` (In Keras/Frameworks): The network automatically calculates and applies the exact amount of zero-padding required to ensure the output feature map has the exact same spatial size as the original input image.


### Stride & Strided Convolutions:
Stride defines the step size or rate at which the kernel shifts across the image.
- Standard Setting: The default stride is 1, meaning the filter slides forward pixel-by-pixel.
- Strided Convolution (Stride > 1): Setting a higher stride (like 2) forces the filter to skip pixels (jumping two steps at a time).
  - The Benefit: It focuses heavily on extracting higher-level abstract features while aggressively ignoring redundant low-level noise.
  - The Computational Advantage: It acts as an immediate downsampler, reducing the size of the feature map, which requires drastically less computing power.
- The Non-Divisible Fraction Issue: A critical issue arises when the image dimensions cannot be cleanly divided by your stride. For instance, if you have 7 columns and attempt a stride of 2, the kernel will complete 2 clean steps but fail to make the third step because it runs out of pixels. Frameworks handle this by either dropping the remaining pixels completely or padding them out manually.







### Pooling Layers (Downsampling & Variance Control):
While convolutional layers are excellent, large feature maps introduce two massive vulnerabilities: they take up immense memory/space, and they suffer from Translation Variance (meaning a feature detected on the far right side of an image looks entirely different to the network than the same feature sitting on the far left side).

**The Fix: Pooling**

Pooling layers act as an aggressive downsampling filter. They slide across the feature map (typically using a window size of $2 \times 2$ and a stride of 2) to condense spatial areas.
- Max Pooling (Most Popular): It scans a patch and extracts only the highest pixel value, printing it to the new grid. It keeps the sharpest, most dominant signals while deleting the rest.
- Other Types: Average Pooling (calculates the mean), Min Pooling, and $L_{2}$Pooling.

**The Structural Impact of Max Pooling ($2 \times 2$, Stride 2):**

- It immediately reduces both the height and width of your feature map by 50%, throwing away 75% of redundant pixels to save computational energy.
- It creates Translation Invariance: Because it only grabs the maximum presence of a feature within a local zone, it will recognize an object (like a cat's ear) even if it shifts slightly to the left, right, up, or down.

### The Blueprint of Iconic CNN Architectures

The effectiveness of a CNN depends heavily on its architectural design: how many convolution layers are stacked, where pooling is placed, and what strides are utilized. Instead of guessing a layout from scratch, modern deep learning builds upon established, battle-tested foundations:

- LeNet: The historic pioneer; designed by Yann LeCun for handwritten digit recognition (MNIST).
- AlexNet: The breakthrough model that popularized deep CNNs by crushing the ImageNet competition in 2012 using GPUs and ReLU.
- VGGNet (VGG16/VGG19): Celebrated for its beautiful simplicity, proving that stacking simple $3 \times 3$ convolutions deeper and deeper yields incredible accuracy.
- GoogLeNet / Inception: Introduced "Inception modules," which run multiple different filter sizes ($1 \times 1, 3 \times 3, 5 \times 5$) in parallel at the exact same layer to catch multi-scale details.
- ResNet (Residual Networks): Solved the vanishing gradient problem in ultra-deep networks by introducing "skip connections" (shortcuts), allowing models to cleanly train over 150 layers deep.

### Summary Checklist for Code Construction
When you begin writing your PyTorch or Keras code blocks in the next section, remember that the classic CNN pipeline follows this strict pattern over and over:

$$\mathbf{Input}\longrightarrow [\mathbf{Conv2D}\longrightarrow \mathbf{ReLU}\longrightarrow \mathbf{MaxPool2D}]\times N\longrightarrow \mathbf{Flatten}\longrightarrow \mathbf{Dense/Linear}\longrightarrow \mathbf{Output}$$

Python Code

In [ ]:
import torch
import torch.nn as nn

class ProductionCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()

        # Block 1: Feature Extraction
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Halves spatial dimensions
        )

        # Block 2: Complex Feature Extraction
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Classifier Head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), # Assuming 28x28 input image (e.g., MNIST/Fashion-MNIST)
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.classifier(x)
        return x

# Test with dummy input batch (Batch Size = 8, Channels = 3, Height = 28, Width = 28)
model = ProductionCNN(in_channels=3, num_classes=10)
dummy_input = torch.randn(8, 3, 28, 28)
output = model(dummy_input)
print(f"Output Logits Shape: {output.shape}")  # Output: torch.Size([8, 10])

## Backpropagation Mechanics in Specialized Layers

During the backward pass (backpropagation), derivatives are calculated in reverse order. While standard layer derivatives are straightforward, Flattening and Max Pooling require unique mathematical reshaping and routing tricks:

1. The Flatten Layer Derivative (Reshaping)
    - Forward Pass: A 2D feature map of size $(2 \times 2)$ is unrolled into a 1D column vector of size $(4 \times 1)$.
    - Backward Pass: The incoming gradient vector of size $(4 \times 1)$ is simply bent back (un-flattened) into its original $(2 \times 2)$ matrix shape so it can flow back into the preceding convolutional layers.
  
2. The Max Pooling Layer Derivative (The Gradient Mask)
    - Forward Pass: The layer scans a pixel patch and passes only the maximum value forward. Crucially, the network saves the exact coordinate position (index) of where that maximum value came from.
    - Backward Pass: The incoming gradient value is routed directly back to that specific saved coordinate. All other positions in the patch receive a gradient of 0, because they did not contribute to the forward prediction and should not be modified.

### Core Mathematical Formulas for Convolutional Updates
When updating the weights and biases of a convolutional layer, the final derived calculus formulas translate into simple matrix operations:

1. The Bias Gradient $(\partial L / \partial b)$
Because a single bias scalar (\(b_{\text{conv}}\)) is added evenly across every single coordinate position in a feature map, its final gradient is calculated by summing up every individual element in the incoming error matrix
$$(\delta _{\text{conv}}):\frac{\partial L}{\partial b_{\text{conv}}}=\sum _{i}\sum _{j}\delta _{\text{conv},ij}$$

2. The Weight Gradient Matrix $(\partial L / \partial W)$
Each individual entry in the weight gradient is computed as the cross-correlation between the input image region (X) and the incoming gradient error map
$$\delta _{\text{conv}}:\frac{\partial L}{\partial w_{mn}}=\sum _{i}\sum _{j}\delta _{\text{conv},ij}\cdot X_{i+m,j+n}$$

  - Compact Matrix Form: In code, this entire operation is processed efficiently as a single convolution/cross-correlation operation $(*)$:
  $$\frac{\partial L}{\partial W_{\text{conv}}}=X*\delta _{\text{conv}}$$

### Data Augmentation
 Data Augmentation (Expanding Datasets)Data augmentation is a regularisation technique where a network artificially manufactures new training samples derived from your original data. For image data, this involves applying random transformations such as:
 - Rotation & Reflection (Flipping horizontally/vertically)
 - Zooming in or out
 - Shifting width and height
 - The Benefit: It introduces variation, preventing the network from overfitting to specific pixel placements.

### Transfer Learning
Transfer learning is a machine learning technique where a model trained on a massive, general task is reused as the starting point for a different, specialized task. Instead of initializing a blank model at zero, it leverages pre-learned patterns, saving immense training time and performing exceptionally well on small custom datasets.

There are two main paradigms of transfer learning:
  - Feature Extraction: We completely freeze the feature extractor (all convolutional layers) and only remove, replace, and train the final Dense/Classifier layer.
  - Fine-Tuning: We replace the final Dense layer and unfreeze the last few deep convolutional layers. This allows the model to subtly warp its high-level shape filters to match the unique textures of your new custom images.

### Deep Learning Frameworks: Sequential vs. Functional APIs
When coding models in frameworks like Keras or TensorFlow, you must choose between two distinct structural design methods:
- Sequential API (The Linear Stack): Enforces a strict, unbranched line of layers. Each layer has exactly one input tensor and exactly one output tensor. It is simple to write but cannot handle complex paths.
- Functional API (The Directed Graph): Removes all linear limitations. It treats layers like functions, allowing you to design complex architectures featuring multiple inputs, multiple outputs, shared layers, and branch merges (like the skip connections found in ResNet).